# Лабораторная работа №6
## Реализация взаимодействия компонентов распределенного приложения по модели Pub/Sub
**Вариант 1:** Интернет-магазин (OrderCreated, OrderCancelled, PaymentCompleted | Logger, StockService, NotificationService)

### 1. Модель события (Event) и базовая архитектура

In [ ]:
using System;
using System.Collections.Generic;
using System.Threading.Tasks;

public class Event
{
    public string EventType { get; set; }
    public int EntityId { get; set; }
    public DateTime Timestamp { get; set; }
    public Dictionary<string, object> Data { get; set; } = new();
}

### 2. Реализация Broker и интерфейса ISubscriber

In [ ]:
public interface ISubscriber
{
    string Name { get; }
    List<string> SubscribedEvents { get; }
    Task HandleEventAsync(Event ev);
}

public class MessageBroker
{
    private readonly Dictionary<string, List<ISubscriber>> _subscribers = new();

    public void Subscribe(string eventType, ISubscriber subscriber)
    {
        if (!_subscribers.ContainsKey(eventType))
        {
            _subscribers[eventType] = new List<ISubscriber>();
        }
        if (!_subscribers[eventType].Contains(subscriber))
        {
            _subscribers[eventType].Add(subscriber);
        }
    }

    public async Task PublishAsync(Event ev)
    {
        Console.WriteLine($"[Broker] Получено событие: {ev.EventType} для объекта {ev.EntityId} в {ev.Timestamp:HH:mm:ss.fff}");
        
        if (!_subscribers.ContainsKey(ev.EventType))
        {
            Console.WriteLine($"[Broker] Нет подписчиков на событие {ev.EventType}");
            return;
        }

        foreach (var subscriber in _subscribers[ev.EventType])
        {
            try
            {
                Console.WriteLine($"[Broker] Доставка события {ev.EventType} подписчику {subscriber.Name}...");
                await subscriber.HandleEventAsync(ev);
            }
            catch (Exception ex)
            {
                Console.WriteLine($"[Broker ошибка] Ошибка при обработке события подписчиком {subscriber.Name}: {ex.Message}");
            }
        }
    }
}

### 3. Реализация Publisher

In [ ]:
public class Publisher
{
    private readonly MessageBroker _broker;

    public Publisher(MessageBroker broker)
    {
        _broker = broker;
    }

    public async Task PublishEventAsync(string eventType, int entityId, Dictionary<string, object> data = null)
    {
        var ev = new Event
        {
            EventType = eventType,
            EntityId = entityId,
            Timestamp = DateTime.Now,
            Data = data ?? new()
        };

        Console.WriteLine($"[Publisher] Публикация события {eventType} (EntityId: {entityId})");
        await _broker.PublishAsync(ev);
    }
}

### 4. Реализация Подписчиков (Вариант 1: Интернет-магазин)

In [ ]:
public class LoggerSubscriber : ISubscriber
{
    public string Name => "Logger";
    public List<string> SubscribedEvents => new() { "OrderCreated", "OrderCancelled", "PaymentCompleted" };

    public async Task HandleEventAsync(Event ev)
    {
        await Task.Delay(100);
        Console.WriteLine($"  [Logger] Записано в лог: Событие {ev.EventType}, ID={ev.EntityId} в {ev.Timestamp:HH:mm:ss}");
    }
}

public class StockServiceSubscriber : ISubscriber
{
    public string Name => "StockService";
    public List<string> SubscribedEvents => new() { "OrderCreated", "OrderCancelled" };

    public async Task HandleEventAsync(Event ev)
    {
        await Task.Delay(300);
        Console.WriteLine($"  [StockService] Обновлен склад для события {ev.EventType}, ID={ev.EntityId}");
    }
}

public class NotificationServiceSubscriber : ISubscriber
{
    public string Name => "NotificationService";
    public List<string> SubscribedEvents => new() { "OrderCreated", "PaymentCompleted" };

    public async Task HandleEventAsync(Event ev)
    {
        await Task.Delay(200);
        Console.WriteLine($"  [NotificationService] Отправлено уведомление клиенту по событию {ev.EventType}, ID={ev.EntityId}");
    }
}

### 5. Добавление нового Subscriber (AnalyticsService)

In [ ]:
public class AnalyticsServiceSubscriber : ISubscriber
{
    public string Name => "AnalyticsService";
    public List<string> SubscribedEvents => new() { "OrderCreated", "PaymentCompleted" };

    public async Task HandleEventAsync(Event ev)
    {
        await Task.Delay(150);
        Console.WriteLine($"  [AnalyticsService] Собрана статистика по событию {ev.EventType}, ID={ev.EntityId}");
    }
}

### 6. Демонстрационный запуск (Program)

In [ ]:
class Program
{
    public static async Task Main(string[] args)
    {
        Console.WriteLine("=== Запуск демонстрации модели Pub/Sub (ЛР №6) ===");
        
        var broker = new MessageBroker();
        var publisher = new Publisher(broker);

        var logger = new LoggerSubscriber();
        var stock = new StockServiceSubscriber();
        var notification = new NotificationServiceSubscriber();

        foreach (var evType in logger.SubscribedEvents) broker.Subscribe(evType, logger);
        foreach (var evType in stock.SubscribedEvents) broker.Subscribe(evType, stock);
        foreach (var evType in notification.SubscribedEvents) broker.Subscribe(evType, notification);

        // 1. Публикация OrderCreated
        await publisher.PublishEventAsync("OrderCreated", 125);
        Console.WriteLine();

        // 2. Публикация PaymentCompleted
        await publisher.PublishEventAsync("PaymentCompleted", 125);
        Console.WriteLine();

        // 3. Подключение нового подписчика AnalyticsService без изменения Publisher
        var analytics = new AnalyticsServiceSubscriber();
        foreach (var evType in analytics.SubscribedEvents) broker.Subscribe(evType, analytics);

        Console.WriteLine("--- После добавления AnalyticsService ---");
        await publisher.PublishEventAsync("OrderCreated", 126);
        
        Console.WriteLine("=== Демонстрация завершена ===");
    }
}

### 7. Контрольные вопросы и ответы
1. **Что такое Pub/Sub?** - Модель взаимодействия компонентов, при которой издатель (Publisher) отправляет сообщения без указания конкретных получателей, а брокер (Broker) доставляет их подписчикам (Subscriber) на основе подписок.
2. **Что такое Publisher, Broker, Subscriber, Event?** - Publisher генерирует события; Broker управляет подписками и маршрутизацией; Subscriber обрабатывает события; Event содержит информацию о произошедшем действии.
3. **Что такое слабая связанность компонентов?** - Ситуация, когда компоненты ничего не знают о деталях реализации друг друга, что позволяет добавлять или удалять подписчиков без изменения издателя.
4. **Как обрабатываются отказы?** - Брокер перехватывает исключения в блоке `try-catch` при доставке конкретному подписчику, что позволяет остальным подписчикам продолжить работу без сбоя системы.